# **Rice Models**

# **1. Download**

In [ ]:
from google.colab import files
import os, zipfile, glob, shutil, random, warnings
from pathlib import Path
import numpy as np
warnings.filterwarnings('ignore')

print("Upload your kaggle.json  (Kaggle > Settings > Create New API Token)")
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(uploaded['kaggle.json'])
os.chmod('/root/.kaggle/kaggle.json', 0o600)

KAGGLE_DATASET = 'cristhiansempertegui/dataset-de-arroz-peruano'
os.system(f"kaggle datasets download -d {KAGGLE_DATASET} -p /content/data")

for z in glob.glob('/content/data/*.zip'):
    with zipfile.ZipFile(z, 'r') as zf:
        zf.extractall('/content/data')
    os.remove(z)

os.system("find /content/data -maxdepth 2 -type d -print")

BASE_DIR = Path("/content/data/dataset")
for cls in ["entero", "mancha", "quebrado", "tiza"]:
    folder = BASE_DIR / cls
    print(f"\nContents of {cls}:")
    print(os.listdir(str(folder))[:20])


# ============================================================
# GLOBAL CONSTANTS
# Class order: Whole · Stained · Broken · Chalky
# ============================================================

PREFIX_TO_CLASS = {          # prefix → English class name
    "e": "Whole",
    "m": "Stained",
    "q": "Broken",
    "t": "Chalky",
}

CLASS_PREFIXES = {           # Spanish folder → prefix
    "entero"  : "e",
    "mancha"  : "m",
    "quebrado": "q",
    "tiza"    : "t",
}

CLASS_ORDER = ["Whole", "Stained", "Broken", "Chalky"]

In [2]:
# Shared hyperparameters
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
SEED       = 42
EPOCHS        = 20       # fase 1: más margen antes de early stopping
EPOCHS_FT     = 20       # fase 2: fine-tuning
LR            = 1e-4     # fase 1
LR_FT         = 1e-5     # fase 2 (10x menor para no destruir pesos)


# **2. Consolidation**

In [ ]:
def consolidate_images(base_dir: Path,
                       output_dir: Path,
                       class_prefixes: dict) -> dict:
    """
    Copy all class images into a single flat folder, renaming them
    with a short prefix  (e.g. e1.png, m1.png …).

    Returns  counters : {'prefix': final_count, ...}
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    counters = {p: 1 for p in class_prefixes.values()}

    for class_name, prefix in class_prefixes.items():
        class_folder = base_dir / class_name
        if not class_folder.exists():
            print(f"[WARNING] Folder not found: {class_folder}")
            continue
        for img_path in glob.glob(str(class_folder / "*.png")):
            new_name = f"{prefix}{counters[prefix]}.png"
            shutil.copy2(img_path, output_dir / new_name)
            counters[prefix] += 1

    return counters


ALL_IMAGES = BASE_DIR / "all_images"
counters   = consolidate_images(BASE_DIR, ALL_IMAGES, CLASS_PREFIXES)

print("\nImages consolidated and renamed successfully.")
for class_name, prefix in CLASS_PREFIXES.items():
    eng = PREFIX_TO_CLASS[prefix]
    print(f"  {eng} ({class_name}): {counters[prefix] - 1} images")
print(f"\nTotal: {len(list(ALL_IMAGES.glob('*')))} images in {ALL_IMAGES}")

# **3. Stratified Split**

In [ ]:
def stratified_split(source_dir      : Path,
                     output_dir      : Path,
                     prefix_to_class : dict,
                     train_ratio     : float = 0.70,
                     val_ratio       : float = 0.15,
                     test_ratio      : float = 0.15,
                     seed            : int   = 42) -> dict:
    """
    Split images stratified by class.
    Returns  split_counts : {'train': {'ClassName': n, ...}, ...}
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6

    random.seed(seed)
    np.random.seed(seed)

    splits       = ["train", "val", "test"]
    split_counts = {s: {} for s in splits}

    for split in splits:
        for class_name in prefix_to_class.values():
            (output_dir / split / class_name).mkdir(parents=True, exist_ok=True)

    for prefix, class_name in prefix_to_class.items():
        images  = sorted(source_dir.glob(f"{prefix}*.png"))
        random.shuffle(images)

        n       = len(images)
        n_train = int(n * train_ratio)
        n_val   = int(n * val_ratio)
        n_test  = n - n_train - n_val

        subsets = {
            "train": images[:n_train],
            "val"  : images[n_train : n_train + n_val],
            "test" : images[n_train + n_val :],
        }

        print(f"  {class_name} ({n})  →  "
              f"train={n_train}  val={n_val}  test={n_test}")

        for split, subset in subsets.items():
            dest = output_dir / split / class_name
            for img_path in subset:
                shutil.copy2(img_path, dest / img_path.name)
            split_counts[split][class_name] = len(subset)

    return split_counts


SPLIT_DIR = BASE_DIR / "rice_dataset"

print("\n=== Stratified Split  (70 / 15 / 15) ===\n")
split_counts = stratified_split(
    source_dir      = ALL_IMAGES,
    output_dir      = SPLIT_DIR,
    prefix_to_class = PREFIX_TO_CLASS,
    seed            = SEED,
)

print("\n=== Split Summary ===")
header = f"{'Class':<12}" + "".join(f"{s.upper():>10}" for s in ["train","val","test"])
print(header)
print("-" * len(header))
for cls in CLASS_ORDER:
    row = f"{cls:<12}" + "".join(
        f"{split_counts[s].get(cls,0):>10}" for s in ["train","val","test"])
    print(row)

# **4. Data Augmentation + Balancing for Train Only**

In [ ]:
from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator, img_to_array, load_img, array_to_img,
)

AUGMENTED_TRAIN_DIR = BASE_DIR / "rice_dataset_augmented" / "train"
VAL_DIR             = SPLIT_DIR / "val"
TEST_DIR            = SPLIT_DIR / "test"


def build_augmentation_generator() -> ImageDataGenerator:
    """Return the ImageDataGenerator used for offline data augmentation."""
    return ImageDataGenerator(
        rotation_range     = 25,
        width_shift_range  = 0.15,
        height_shift_range = 0.15,
        shear_range        = 0.15,
        zoom_range         = 0.15,
        horizontal_flip    = True,
        brightness_range   = [0.7, 1.3],
        fill_mode          = 'nearest',
    )


def augment_and_balance_train(train_source_dir : Path,
                               output_dir       : Path,
                               datagen          : ImageDataGenerator,
                               seed             : int = 42) -> dict:
    """
    Copy all training images to output_dir, then generate synthetic
    images for minority classes until every class reaches the count
    of the majority class.

    Returns  final_counts : {'ClassName': count, ...}
    """
    random.seed(seed)
    np.random.seed(seed)

    output_dir.mkdir(parents=True, exist_ok=True)

    class_dirs  = [d for d in train_source_dir.iterdir() if d.is_dir()]
    class_names = [d.name for d in class_dirs]

    for cls in class_names:
        (output_dir / cls).mkdir(parents=True, exist_ok=True)

    original_counts = {
        d.name: len(list(d.glob("*.png"))) for d in class_dirs
    }
    max_count = max(original_counts.values())

    print("\n=== Augmentation & Balancing  (TRAIN ONLY) ===")
    print(f"Target count per class: {max_count}\n")
    print("Original train counts:")
    for cls in CLASS_ORDER:
        print(f"  {cls}: {original_counts.get(cls, 0)}")

    # Copy originals
    print("\nCopying original training images …")
    for cls_dir in class_dirs:
        for img_path in cls_dir.glob("*.png"):
            dest = output_dir / cls_dir.name / img_path.name
            if not dest.exists():
                shutil.copy2(img_path, dest)

    # Generate synthetics for minority classes
    for cls_dir in class_dirs:
        cls_name  = cls_dir.name
        orig_imgs = list(cls_dir.glob("*.png"))
        deficit   = max_count - original_counts[cls_name]

        if deficit <= 0:
            print(f"  {cls_name}: already at maximum, skipping.")
            continue

        print(f"  {cls_name}: generating {deficit} synthetic images "
              f"({original_counts[cls_name]} → {max_count}) …")

        generated = 0
        while generated < deficit:
            img_path = random.choice(orig_imgs)
            img = load_img(img_path)
            x   = img_to_array(img).reshape((1,) + img_to_array(img).shape)
            for _ in datagen.flow(x,
                                  batch_size  = 1,
                                  save_to_dir = str(output_dir / cls_name),
                                  save_prefix = cls_name[0],
                                  save_format = 'png'):
                generated += 1
                if generated >= deficit:
                    break

    final_counts = {
        cls: len(list((output_dir / cls).glob("*.png")))
        for cls in class_names
    }
    return final_counts


datagen_aug         = build_augmentation_generator()
final_train_counts  = augment_and_balance_train(
    train_source_dir = SPLIT_DIR / "train",
    output_dir       = AUGMENTED_TRAIN_DIR,
    datagen          = datagen_aug,
    seed             = SEED,
)

print("\nFinal train counts after augmentation:")
for cls in CLASS_ORDER:
    print(f"  {cls}: {final_train_counts.get(cls, 0)}")
print("\nVal and Test sets are UNCHANGED.")

# **5. Visualization**

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


def plot_class_distribution(split_counts : dict,
                             title        : str = "Class Distribution") -> None:
    """Grouped bar chart – image count per class × split."""
    splits = list(split_counts.keys())
    x      = np.arange(len(CLASS_ORDER))
    width  = 0.25

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, split in enumerate(splits):
        counts = [split_counts[split].get(c, 0) for c in CLASS_ORDER]
        ax.bar(x + i * width, counts, width, label=split.upper())

    ax.set_xticks(x + width)
    ax.set_xticklabels(CLASS_ORDER)
    ax.set_title(title)
    ax.set_xlabel("Class")
    ax.set_ylabel("Number of images")
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()


def show_augmentation_comparison(original_dir : Path,
                                  prefix       : str,
                                  class_name   : str,
                                  datagen      : ImageDataGenerator,
                                  n_rows       : int = 2) -> None:
    """Show n_rows originals alongside 3 augmented versions each."""
    images = list(original_dir.glob(f"{prefix}*.png"))
    if not images:
        print(f"No images found for prefix '{prefix}'")
        return

    fig, axes = plt.subplots(n_rows, 4, figsize=(14, n_rows * 3))
    fig.suptitle(f"Augmentation preview – {class_name}", fontsize=14)
    for i in range(n_rows):
        img      = load_img(random.choice(images))
        axes[i, 0].imshow(img)
        axes[i, 0].set_title("Original")
        axes[i, 0].axis("off")
        x        = img_to_array(img).reshape((1,) + img_to_array(img).shape)
        aug_iter = datagen.flow(x, batch_size=1)
        for j in range(1, 4):
            batch = next(aug_iter)
            axes[i, j].imshow(array_to_img(batch[0]))
            axes[i, j].set_title(f"Augmented {j}")
            axes[i, j].axis("off")
    plt.tight_layout()
    plt.show()


plot_class_distribution(split_counts,
    title="Class Distribution per Split (before augmentation)")

for prefix, class_name in PREFIX_TO_CLASS.items():
    show_augmentation_comparison(ALL_IMAGES, prefix, class_name,
                                 datagen_aug, n_rows=2)

# **6. Image Integrity Check**

In [ ]:
from PIL import Image
from tqdm import tqdm


def verify_and_clean_images(directories: list) -> int:
    """Open every image; remove corrupt files. Returns count removed."""
    total_removed = 0
    for directory in directories:
        print(f"\nChecking {directory} …")
        corrupt = []
        for img_path in tqdm(list(directory.rglob("*.*")), desc="Verifying"):
            try:
                with Image.open(img_path) as img:
                    img.verify()
            except Exception:
                print(f"  Corrupt: {img_path}")
                corrupt.append(img_path)
        for img_path in corrupt:
            img_path.unlink()
            print(f"  Removed: {img_path}")
        total_removed += len(corrupt)

    print(f"\nIntegrity check complete.  Removed: {total_removed} file(s).")
    return total_removed


verify_and_clean_images([AUGMENTED_TRAIN_DIR, VAL_DIR, TEST_DIR])

# **7. Model Training**

In [ ]:
# Paths used:
#   train → AUGMENTED_TRAIN_DIR   (balanced after augmentation)
#   val   → VAL_DIR               (original imbalance, no aug)
#   test  → TEST_DIR              (original imbalance, no aug)
#
# Classification head (same for all):
#   GAP → Dense(256,relu) → BN → Dropout(0.4)
#       → Dense(128,relu) → Dropout(0.3) → Dense(4,softmax)
#
# Callbacks — Phase 1 (head only):
#   EarlyStopping     : monitor=val_loss  patience=3  restore_best_weights=True
#   ReduceLROnPlateau : monitor=val_loss  factor=0.3  patience=2  min_lr=1e-6
#
# Callbacks — Phase 2 (fine-tuning):
#   EarlyStopping     : monitor=val_loss  patience=5  restore_best_weights=True
#   ReduceLROnPlateau : monitor=val_loss  factor=0.5  patience=3  min_lr=1e-7
#
# Preprocessing (all models use backbone-specific preprocess_input):
#   MobileNetV2    → mobilenet_v2.preprocess_input
#   EfficientNetB0 → efficientnet.preprocess_input
#   ResNet50       → resnet50.preprocess_input
#   DenseNet121    → densenet.preprocess_input
#   InceptionV3    → inception_v3.preprocess_input
#
# Special preprocessing:
#   EfficientNetB0 → efficientnet.preprocess_input (no external rescale)
#   InceptionV3    → inception_v3.preprocess_input (backbone training=False)
# ============================================================

import tensorflow as tf
from tensorflow.keras import Model, layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator as IDG
from tensorflow.keras.callbacks import (EarlyStopping, ModelCheckpoint, ReduceLROnPlateau)
from tensorflow.keras.layers import (GlobalAveragePooling2D, Dense, Dropout, BatchNormalization, Input)
from tensorflow.keras.applications import (MobileNetV2, EfficientNetB0, ResNet50, DenseNet121, InceptionV3)
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess
from tensorflow.keras.applications.inception_v3 import preprocess_input as inc_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_preprocess
from tensorflow.keras.applications.resnet50     import preprocess_input as res_preprocess
from tensorflow.keras.applications.densenet     import preprocess_input as den_preprocess
from sklearn.metrics import classification_report, confusion_matrix

TRAIN_PATH = str(AUGMENTED_TRAIN_DIR)
VAL_PATH   = str(VAL_DIR)
TEST_PATH  = str(TEST_DIR)


# ── Shared callbacks ────────────────────────────────────────

def get_callbacks_phase1(model_name: str) -> list:
    """Fase 1 — head only. ES agresivo, guarda el mejor."""
    return [
        EarlyStopping(monitor='val_loss', patience=3,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                          patience=2, min_lr=1e-6, verbose=1),
        ModelCheckpoint(f'best_{model_name}_p1.keras',
                        monitor='val_accuracy',
                        save_best_only=True, verbose=0),
    ]

def get_callbacks_phase2(model_name: str) -> list:
    """Fase 2 — fine-tuning. ES con más paciencia, sin restore para
    dejar que los pesos del backbone se adapten gradualmente."""
    return [
        EarlyStopping(monitor='val_loss', patience=5,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=3, min_lr=1e-7, verbose=1),
        ModelCheckpoint(f'best_{model_name}_p2.keras',
                        monitor='val_accuracy',
                        save_best_only=True, verbose=0),
    ]


# ── Shared classification head ──────────────────────────────

def add_classification_head(base_output, num_classes: int):
    """
    Shared head from team notebook:
    GAP → Dense(256,relu) → BN → Dropout(0.4)
        → Dense(128,relu) → Dropout(0.3) → Dense(n,softmax)
    """
    x = GlobalAveragePooling2D()(base_output)
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    return Dense(num_classes, activation='softmax')(x)


# ── Data generators ─────────────────────────────────────────

def make_generator(preprocess_fn=None, rescale_val: float = None):
    """Return an ImageDataGenerator with the appropriate preprocessing."""
    if preprocess_fn is not None:
        return IDG(preprocessing_function=preprocess_fn)
    return IDG(rescale=rescale_val or 1.0 / 255)


def flow_dirs(gen, img_size=IMG_SIZE, batch_size=BATCH_SIZE,
              seed=SEED, class_order=CLASS_ORDER):
    """Return (train_gen, val_gen, test_gen) from a given generator."""
    train = gen.flow_from_directory(
        TRAIN_PATH, target_size=img_size, batch_size=batch_size,
        class_mode='categorical', classes=class_order,
        shuffle=True, seed=seed)
    val = gen.flow_from_directory(
        VAL_PATH, target_size=img_size, batch_size=batch_size,
        class_mode='categorical', classes=class_order,
        shuffle=False, seed=seed)
    test = gen.flow_from_directory(
        TEST_PATH, target_size=img_size, batch_size=batch_size,
        class_mode='categorical', classes=class_order,
        shuffle=False, seed=seed)
    return train, val, test


# ── Training + evaluation helpers ───────────────────────────

def plot_history(history, model_name: str) -> None:
    """Accuracy and Loss curves (train vs validation) — two separate figures."""
    acc      = history.history['accuracy']
    val_acc  = history.history['val_accuracy']
    loss     = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_r = range(1, len(acc) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs_r, acc,     color='steelblue',  linewidth=2,
             marker='o', markersize=4, label='Training Accuracy')
    plt.plot(epochs_r, val_acc, color='darkorange', linewidth=2,
             marker='s', markersize=4, label='Validation Accuracy')
    plt.title(f'{model_name} — Accuracy: Training vs Validation',
              fontsize=13, fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend(loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs_r, loss,     color='steelblue',  linewidth=2,
             marker='o', markersize=4, label='Training Loss')
    plt.plot(epochs_r, val_loss, color='darkorange', linewidth=2,
             marker='s', markersize=4, label='Validation Loss')
    plt.title(f'{model_name} — Loss: Training vs Validation',
              fontsize=13, fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend(loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()


def run_evaluation(model, test_gen, model_name: str,
                   class_order: list = CLASS_ORDER) -> dict:
    """
    Evaluate model on test_gen.
    Prints accuracy, loss, classification report, confusion matrix.
    Returns {'y_true', 'y_pred', 'accuracy', 'loss', 'report'}.
    """
    test_gen.reset()
    test_loss, test_acc = model.evaluate(test_gen, verbose=0)
    print(f"\n{'='*55}")
    print(f"  {model_name} — FINAL TEST RESULTS")
    print(f"{'='*55}")
    print(f"  Test Accuracy : {test_acc * 100:.2f}%")
    print(f"  Test Loss     : {test_loss:.4f}")
    print(f"{'='*55}")

    test_gen.reset()
    y_probs = model.predict(test_gen, verbose=0)
    y_pred  = np.argmax(y_probs, axis=1)
    y_true  = test_gen.classes

    print(f"\nClassification Report ({model_name}):")
    print(classification_report(y_true, y_pred, target_names=class_order))

    cm = confusion_matrix(y_true, y_pred,
                          labels=list(range(len(class_order))))
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_order, yticklabels=class_order)
    plt.title(f'Confusion Matrix — {model_name} (Test Set)')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()

    return {'y_true'  : y_true,
            'y_pred'  : y_pred,
            'accuracy': test_acc,
            'loss'    : test_loss,
            'report'  : classification_report(y_true, y_pred,
                            target_names=class_order, output_dict=True)}


# ── Fine-tuning helper (identical for all models) ────

def apply_fine_tuning(model, base_model, unfreeze_from: int,
                      lr_ft: float = LR_FT) -> None:
    """
    Fase 2: congela todas las capas del backbone excepto las últimas
    `abs(unfreeze_from)`. Recompila con LR muy bajo.
    Modifica el modelo in-place.
    """
    for layer in base_model.layers[:unfreeze_from]:
        layer.trainable = False
    for layer in base_model.layers[unfreeze_from:]:
        layer.trainable = True

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_ft),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )


# ── Class weights helper ────────────────────────────────────
from sklearn.utils.class_weight import compute_class_weight

def get_class_weights(train_gen) -> dict:
    """
    Pesos inversamente proporcionales a la frecuencia de cada clase.
    Penaliza más los errores en clases minoritarias (especialmente Chalky).
    Aplicado de forma idéntica a todos los modelos.
    """
    classes = train_gen.classes
    weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(classes),
        y=classes,
    )
    return dict(enumerate(weights))

# ── Storage for results of all models ───────────────────────
all_results = {}   # {'ModelName': {'y_true', 'y_pred', 'accuracy', ...}}

# **7.1. MobileNetV2 Model**

In [ ]:
print("\n" + "="*60)
print("  MODEL 1 — MobileNetV2")
print("="*60)

gen_mob                      = make_generator(preprocess_fn=mob_preprocess)
train_mob, val_mob, test_mob = flow_dirs(gen_mob)
NUM_CLASSES                  = train_mob.num_classes
class_weights_mob            = get_class_weights(train_mob)

base_mob           = MobileNetV2(input_shape=IMG_SIZE+(3,),
                                 include_top=False, weights='imagenet')
base_mob.trainable = False

model_mobilenet = Model(inputs=base_mob.input,
                        outputs=add_classification_head(
                            base_mob.output, NUM_CLASSES))
model_mobilenet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
    loss='categorical_crossentropy', metrics=['accuracy'])

# ── Fase 1: solo el head ─────────────────────────────────────
print("\n--- Phase 1: head only ---")
history_mob_p1 = model_mobilenet.fit(
    train_mob, epochs=EPOCHS, validation_data=val_mob,
    callbacks=get_callbacks_phase1('mobilenetv2_p1'),
    class_weight=class_weights_mob)

# ── Fase 2: fine-tuning ─────────────────────
print("\n--- Phase 2: fine-tuning  ---")
apply_fine_tuning(model_mobilenet, base_mob, unfreeze_from=-10)
history_mob_p2 = model_mobilenet.fit(
    train_mob, epochs=EPOCHS_FT, validation_data=val_mob,
    callbacks=get_callbacks_phase2('mobilenetv2_p2'),
    class_weight=class_weights_mob)

plot_history(history_mob_p1, 'MobileNetV2 — Phase 1')
plot_history(history_mob_p2, 'MobileNetV2 — Phase 2 (Fine-tuning)')
all_results['MobileNetV2'] = run_evaluation(
    model_mobilenet, test_mob, 'MobileNetV2')

# **7.2. EfficientNetB0 Model**

In [ ]:
print("\n" + "="*60)
print("  MODEL 2 — EfficientNetB0")
print("="*60)

gen_eff                      = make_generator(preprocess_fn=eff_preprocess)
train_eff, val_eff, test_eff = flow_dirs(gen_eff)
class_weights_eff            = get_class_weights(train_eff)

base_eff           = EfficientNetB0(input_shape=IMG_SIZE+(3,),
                                    include_top=False, weights='imagenet')
base_eff.trainable = False

model_efficient = Model(inputs=base_eff.input,
                        outputs=add_classification_head(
                            base_eff.output, NUM_CLASSES))
model_efficient.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
    loss='categorical_crossentropy', metrics=['accuracy'])

# ── Fase 1 ───────────────────────────────────────────────────
print("\n--- Phase 1: head only ---")
history_eff_p1 = model_efficient.fit(
    train_eff, epochs=EPOCHS, validation_data=val_eff,
    callbacks=get_callbacks_phase1('efficientnetb0_p1'),
    class_weight=class_weights_eff)

# ── Fase 2: fine-tuning ─────────────────────
print("\n--- Phase 2: fine-tuning ---")
apply_fine_tuning(model_efficient, base_eff, unfreeze_from=-10)
history_eff_p2 = model_efficient.fit(
    train_eff, epochs=EPOCHS_FT, validation_data=val_eff,
    callbacks=get_callbacks_phase2('efficientnetb0_p2'),
    class_weight=class_weights_eff)

plot_history(history_eff_p1, 'EfficientNetB0 — Phase 1')
plot_history(history_eff_p2, 'EfficientNetB0 — Phase 2 (Fine-tuning)')
all_results['EfficientNetB0'] = run_evaluation(
    model_efficient, test_eff, 'EfficientNetB0')

# **7.3. ResNet50 Model**

In [ ]:
print("\n" + "="*60)
print("  MODEL 3 — ResNet50")
print("="*60)

gen_res                      = make_generator(preprocess_fn=res_preprocess)
train_res, val_res, test_res = flow_dirs(gen_res)
class_weights_res            = get_class_weights(train_res)

base_res           = ResNet50(input_shape=IMG_SIZE+(3,),
                              include_top=False, weights='imagenet')
base_res.trainable = False

model_resnet = Model(inputs=base_res.input,
                     outputs=add_classification_head(
                         base_res.output, NUM_CLASSES))
model_resnet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
    loss='categorical_crossentropy', metrics=['accuracy'])

# ── Fase 1 ───────────────────────────────────────────────────
print("\n--- Phase 1: head only ---")
history_res_p1 = model_resnet.fit(
    train_res, epochs=EPOCHS, validation_data=val_res,
    callbacks=get_callbacks_phase1('resnet50_p1'),
    class_weight=class_weights_res)

# ── Fase 2: fine-tuning ─────────────────────
print("\n--- Phase 2: fine-tuning ---")
apply_fine_tuning(model_resnet, base_res, unfreeze_from=-10)
history_res_p2 = model_resnet.fit(
    train_res, epochs=EPOCHS_FT, validation_data=val_res,
    callbacks=get_callbacks_phase2('resnet50_p2'),
    class_weight=class_weights_res)

plot_history(history_res_p1, 'ResNet50 — Phase 1')
plot_history(history_res_p2, 'ResNet50 — Phase 2 (Fine-tuning)')
all_results['ResNet50'] = run_evaluation(
    model_resnet, test_res, 'ResNet50')

# **7.4. DenseNet121 Model**

In [ ]:
print("\n" + "="*60)
print("  MODEL 4 — DenseNet121")
print("="*60)

gen_den                      = make_generator(preprocess_fn=den_preprocess)
train_den, val_den, test_den = flow_dirs(gen_den)
class_weights_den            = get_class_weights(train_den)

base_den           = DenseNet121(input_shape=IMG_SIZE+(3,),
                                 include_top=False, weights='imagenet')
base_den.trainable = False

model_densenet = Model(inputs=base_den.input,
                       outputs=add_classification_head(
                           base_den.output, NUM_CLASSES))
model_densenet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
    loss='categorical_crossentropy', metrics=['accuracy'])

# ── Fase 1 ───────────────────────────────────────────────────
print("\n--- Phase 1: head only ---")
history_den_p1 = model_densenet.fit(
    train_den, epochs=EPOCHS, validation_data=val_den,
    callbacks=get_callbacks_phase1('densenet121_p1'),
    class_weight=class_weights_den)

# ── Fase 2: fine-tuning  ─────────────────────
print("\n--- Phase 2: fine-tuning ---")
apply_fine_tuning(model_densenet, base_den, unfreeze_from=-10)
history_den_p2 = model_densenet.fit(
    train_den, epochs=EPOCHS_FT, validation_data=val_den,
    callbacks=get_callbacks_phase2('densenet121_p2'),
    class_weight=class_weights_den)

plot_history(history_den_p1, 'DenseNet121 — Phase 1')
plot_history(history_den_p2, 'DenseNet121 — Phase 2 (Fine-tuning)')
all_results['DenseNet121'] = run_evaluation(
    model_densenet, test_den, 'DenseNet121')

# **7.5. InceptionV3 Model**

In [ ]:
print("\n" + "="*60)
print("  MODEL 5 — InceptionV3")
print("="*60)

gen_inc                      = make_generator(preprocess_fn=inc_preprocess)
train_inc, val_inc, test_inc = flow_dirs(gen_inc)
class_weights_inc            = get_class_weights(train_inc)

base_inc           = InceptionV3(input_shape=IMG_SIZE+(3,),
                                 include_top=False, weights='imagenet')
base_inc.trainable = False

inp = Input(shape=IMG_SIZE + (3,))
x   = base_inc(inp, training=False)
out = add_classification_head(x, NUM_CLASSES)

model_inception = Model(inputs=inp, outputs=out)
model_inception.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
    loss='categorical_crossentropy', metrics=['accuracy'])

# ── Fase 1 ───────────────────────────────────────────────────
print("\n--- Phase 1: head only ---")
history_inc_p1 = model_inception.fit(
    train_inc, epochs=EPOCHS, validation_data=val_inc,
    callbacks=get_callbacks_phase1('inceptionv3_p1'),
    class_weight=class_weights_inc)

# ── Fase 2: fine-tuning ─────────────────────
print("\n--- Phase 2: fine-tuning ---")
apply_fine_tuning(model_inception, base_inc, unfreeze_from=-10)
history_inc_p2 = model_inception.fit(
    train_inc, epochs=EPOCHS_FT, validation_data=val_inc,
    callbacks=get_callbacks_phase2('inceptionv3_p2'),
    class_weight=class_weights_inc)

plot_history(history_inc_p1, 'InceptionV3 — Phase 1')
plot_history(history_inc_p2, 'InceptionV3 — Phase 2 (Fine-tuning)')
all_results['InceptionV3'] = run_evaluation(
    model_inception, test_inc, 'InceptionV3')

# **8. Model Comparison**

In [ ]:
def print_comparison_table(all_results: dict) -> None:
    """Print accuracy and loss comparison across all models."""
    MODEL_ORDER = ["MobileNetV2", "EfficientNetB0",
                   "ResNet50", "DenseNet121", "InceptionV3"]

    print(f"\n{'='*55}")
    print("  OVERALL COMPARISON — Test Set")
    print(f"{'='*55}")
    print(f"{'Model':<16} {'Accuracy (%)':>14} {'Loss':>10}")
    print("-" * 42)
    for name in MODEL_ORDER:
        r = all_results[name]
        print(f"{name:<16} {r['accuracy']*100:>14.2f} {r['loss']:>10.4f}")

    print(f"\n{'Per-class F1-scores':^55}")
    print(f"{'Model':<16}" +
          "".join(f"{cls:>12}" for cls in CLASS_ORDER))
    print("-" * (16 + 12 * len(CLASS_ORDER)))
    for name in MODEL_ORDER:
        rep = all_results[name]['report']
        row = f"{name:<16}"
        for cls in CLASS_ORDER:
            row += f"{rep[cls]['f1-score']:>12.4f}"
        print(row)


print_comparison_table(all_results)

# **9. Confidence Intervals**

In [ ]:
# ============================================================
# Formula:
#   p  = accuracy
#   SE = sqrt( p*(1-p) / n )
#   IC95% = p ± 1.96 * SE
# ============================================================

def binomial_ci(accuracy: float,
                n_test  : int,
                z       : float = 1.96) -> tuple:
    """
    Normal approximation binomial CI.
        p  = accuracy
        SE = sqrt(p*(1-p)/n)
        CI = p ± z*SE

    Returns (lower, upper) as percentages.
    """
    p  = accuracy
    se = np.sqrt(p * (1 - p) / n_test)
    lo = max(0.0, p - z * se)
    hi = min(1.0, p + z * se)
    return lo * 100, hi * 100


def print_confidence_intervals(all_results: dict) -> None:
    """Print 95% CI for each model using the binomial normal approximation."""
    MODEL_ORDER = ["MobileNetV2", "EfficientNetB0",
                   "ResNet50", "DenseNet121", "InceptionV3"]

    # n_test is the same for all models (same fixed test split)
    n_test = len(all_results[MODEL_ORDER[0]]['y_true'])

    print(f"\n{'='*65}")
    print(f"  95% Confidence Intervals  (Binomial normal approximation)")
    print(f"  Formula: p ± 1.96 × √(p(1−p)/n)   |  n = {n_test} test images")
    print(f"{'='*65}")
    print(f"{'Model':<16} {'Accuracy':>10} {'95% CI Lower':>14} "
          f"{'95% CI Upper':>14} {'Interval':>20}")
    print("-" * 76)

    for name in MODEL_ORDER:
        acc    = all_results[name]['accuracy']
        lo, hi = binomial_ci(acc, n_test)
        print(f"{name:<16} {acc*100:>10.2f}%  [{lo:>6.2f}%,  {hi:>6.2f}%]"
              f"     {hi-lo:>6.2f} pp wide")

    # Visual bar chart
    fig, ax = plt.subplots(figsize=(9, 5))
    x     = np.arange(len(MODEL_ORDER))
    accs  = [all_results[n]['accuracy'] * 100 for n in MODEL_ORDER]
    lows  = [accs[i] - binomial_ci(all_results[n]['accuracy'], n_test)[0]
             for i, n in enumerate(MODEL_ORDER)]
    highs = [binomial_ci(all_results[n]['accuracy'], n_test)[1] - accs[i]
             for i, n in enumerate(MODEL_ORDER)]

    ax.bar(x, accs, color='steelblue', alpha=0.7, zorder=2)
    ax.errorbar(x, accs, yerr=[lows, highs],
                fmt='none', color='black', capsize=6, linewidth=2, zorder=3)
    ax.set_xticks(x)
    ax.set_xticklabels(MODEL_ORDER, rotation=15, ha='right')
    ax.set_xlabel("Model")
    ax.set_ylabel("Test Accuracy (%)")
    ax.set_title("Test Accuracy with 95% Binomial CI\n"
                 "p ± 1.96 × √(p(1−p)/n)")
    ax.set_ylim(max(0, min(accs) - 5), min(100, max(accs) + 5))
    ax.grid(axis='y', linestyle='--', alpha=0.4, zorder=1)
    plt.tight_layout()
    plt.show()


print_confidence_intervals(all_results)

# **10. McNemar Comparison**

In [ ]:
# ============================================================
# All 10 pairs from 5 models.
# H0: both classifiers make the same errors on the same test images.
# Uses exact binomial McNemar (statsmodels).
#
# Contingency table:
#   b = A correct, B wrong
#   c = A wrong,   B correct
# Statistic (exact):  min(b,c) follows Binomial(b+c, 0.5) under H0
# ============================================================

from itertools import combinations
from statsmodels.stats.contingency_tables import mcnemar as mcnemar_test_fn


def mcnemar_pair(y_true   : np.ndarray,
                 y_pred_a : np.ndarray,
                 y_pred_b : np.ndarray,
                 exact    : bool = True) -> dict:
    """
    McNemar test between two classifiers on the same y_true.

    b = A correct, B wrong
    c = A wrong,   B correct
    exact=True → exact binomial;  exact=False → chi-square
    """
    correct_a = (y_pred_a == y_true)
    correct_b = (y_pred_b == y_true)
    b = int(np.sum( correct_a & ~correct_b))
    c = int(np.sum(~correct_a &  correct_b))

    result = mcnemar_test_fn([[0, b], [c, 0]],
                              exact=exact, correction=False)
    return {
        'b'          : b,
        'c'          : c,
        'statistic'  : result.statistic,
        'p_value'    : result.pvalue,
        'significant': result.pvalue < 0.05,
    }


def run_mcnemar_all_pairs(all_results : dict) -> pd.DataFrame:
    """Run McNemar test for every model pair. Returns result DataFrame."""
    MODEL_ORDER = ["MobileNetV2", "EfficientNetB0",
                   "ResNet50", "DenseNet121", "InceptionV3"]

    y_true = all_results[MODEL_ORDER[0]]['y_true']   # same for all models
    rows   = []

    for name_a, name_b in combinations(MODEL_ORDER, 2):
        res = mcnemar_pair(y_true,
                           all_results[name_a]['y_pred'],
                           all_results[name_b]['y_pred'],
                           exact=True)
        rows.append({
            'Model A'        : name_a,
            'Model B'        : name_b,
            'b (A✓ B✗)'      : res['b'],
            'c (A✗ B✓)'      : res['c'],
            'Statistic'      : round(res['statistic'], 4),
            'p-value'        : round(res['p_value'],   6),
            'Significant'    : 'Yes' if res['significant'] else 'No',
        })

    return pd.DataFrame(rows)


def plot_mcnemar_heatmap(mcnemar_df  : pd.DataFrame,
                         model_order : list) -> None:
    """Lower-triangular heatmap of McNemar p-values."""
    n   = len(model_order)
    idx = {m: i for i, m in enumerate(model_order)}
    mat = np.ones((n, n))

    for _, row in mcnemar_df.iterrows():
        i = idx[row['Model A']]
        j = idx[row['Model B']]
        mat[i, j] = row['p-value']
        mat[j, i] = row['p-value']

    mask = np.eye(n, dtype=bool)

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(mat, annot=True, fmt='.4f',
                xticklabels=model_order, yticklabels=model_order,
                cmap='YlOrRd_r', vmin=0, vmax=0.1,
                mask=mask, ax=ax, linewidths=0.5)
    ax.set_title("McNemar p-values  (all model pairs)\n"
                 "Darker = smaller p-value  |  α = 0.05")
    ax.set_xlabel("Model B")
    ax.set_ylabel("Model A")
    plt.tight_layout()
    plt.show()


MODEL_ORDER = ["MobileNetV2", "EfficientNetB0",
               "ResNet50", "DenseNet121", "InceptionV3"]

mcnemar_df = run_mcnemar_all_pairs(all_results)

print(f"\n{'='*70}")
print("  McNemar Pairwise Tests  (all 10 pairs, exact binomial, α=0.05)")
print(f"{'='*70}")
print(mcnemar_df.to_markdown(index=False, tablefmt='simple', floatfmt='.6f'))

sig = mcnemar_df[mcnemar_df['Significant'] == 'Yes']
print(f"\nSignificant pairs: {len(sig)} / {len(mcnemar_df)}")
if not sig.empty:
    print(sig[['Model A', 'Model B', 'p-value']].to_string(index=False))
else:
    print("  No statistically significant differences found at α=0.05.")

plot_mcnemar_heatmap(mcnemar_df, MODEL_ORDER)

# **11. Consolidated Results Table**

In [ ]:
def plot_consolidated_table(all_results: dict) -> None:
    MODEL_ORDER = ["MobileNetV2", "EfficientNetB0",
                   "ResNet50", "DenseNet121", "InceptionV3"]

    n_test = len(all_results[MODEL_ORDER[0]]['y_true'])

    # ── Recopilar datos ──────────────────────────────────────
    rows = []
    for name in MODEL_ORDER:
        r      = all_results[name]
        acc    = r['accuracy'] * 100
        loss   = r['loss']
        rep    = r['report']
        lo, hi = binomial_ci(r['accuracy'], n_test)

        rows.append({
            'Model'        : name,
            'Accuracy (%)' : round(acc, 2),
            'Loss'         : round(loss, 4),
            'CI 95%'       : f"[{lo:.2f}%, {hi:.2f}%]",
            'F1 Whole'     : round(rep['Whole']['f1-score'],    4),
            'F1 Stained'   : round(rep['Stained']['f1-score'],  4),
            'F1 Broken'    : round(rep['Broken']['f1-score'],   4),
            'F1 Chalky'    : round(rep['Chalky']['f1-score'],   4),
            'Macro F1'     : round(rep['macro avg']['f1-score'],4),
        })

    df = pd.DataFrame(rows).set_index('Model')

    # ── Rank por accuracy ────────────────────────────────────
    df['Rank'] = df['Accuracy (%)'].rank(ascending=False).astype(int)
    df = df.sort_values('Rank')

    print(f"\n{'='*90}")
    print("  CONSOLIDATED RESULTS — All Models (Test Set)")
    print(f"  n_test = {n_test} images  |  Split 70-15-15  |  ImageNet weights (frozen)")
    print(f"{'='*90}")
    print(df.to_string())
    print(f"{'='*90}")

    # ── Heatmap visual ───────────────────────────────────────
    numeric_cols = ['Accuracy (%)', 'F1 Whole', 'F1 Stained',
                    'F1 Broken', 'F1 Chalky', 'Macro F1']
    heat_df = df[numeric_cols]

    fig, ax = plt.subplots(figsize=(11, 4))
    sns.heatmap(
        heat_df,
        annot      = True,
        fmt        = '.4g',
        cmap       = 'YlGn',
        linewidths = 0.5,
        ax         = ax,
        vmin       = heat_df.min().min(),
        vmax       = heat_df.max().max(),
    )
    ax.set_title(
        "Consolidated Model Comparison — Accuracy & F1 per Class (Test Set)\n"
        f"Split 70-15-15  |  n_test={n_test}  |  Two-phase transfer learning (last 10 layers unfrozen)"
        fontsize=12, fontweight='bold'
    )
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.show()


plot_consolidated_table(all_results)